# 2D classification with physical SYNE-KANs

This is the public reproduction notebook for the two-dimensional classification experiments in *Learning Nonlinear Heterogeneity in Physical Kolmogorov-Arnold Networks*.

The released tasks are the 5×5 checkerboard and the five-circle yin–yang problem. Both are trained using the experimentally measured B1 SYNE digital twin as the nonlinear physical substrate. The twin remains frozen throughout: we train the physical ranges, control voltages and output coefficients around it, rather than replacing the device with an analytic activation.

The development Optuna sweeps have already been completed. They are not rerun here. This notebook contains the two fixed paper KANs, the frozen MLP optimiser recipe, the full small-MLP size sweep, fresh evaluation seeds, resumable result logging and the final decision-boundary plots.

Two run profiles are supplied:

- `quick`: a short end-to-end installation check;
- `paper`: the complete three-repeat KAN and MLP comparison.

The result CSV is written after every completed model, so an interrupted paper sweep can be resumed by rerunning the notebook.

## Before running

Place the frozen B1 digital-twin checkpoint beside this notebook as:

`B1_MLP_3_50_50_1.pt`

Alternatively, set the environment variable `SYNE_TWIN_PATH` to the checkpoint location. Results are written beneath `results_2d_classification_release_<profile>/` unless `SYNE_RESULTS_DIR` is set.

The notebook requires Python 3.9+, PyTorch, NumPy, pandas and Matplotlib. CUDA is used when available; CPU remains a valid, slower fallback.

Start with `RUN_PROFILE = "quick"`. The paper profile contains 174 training runs when both model families are enabled, so it is worth checking the complete pipeline before committing the machine to it.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")


def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def stable_seed(*parts) -> int:
    """Build a repeatable integer seed from a readable run description."""
    raw = "|".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:8], 16)


print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

## Run configuration

The two KAN architectures are fixed separately because the tasks do not need the same network. The checkerboard uses `[2,6,1]`; the compact yin–yang model uses `[2,2,1]`. Both use eight SYNE devices per edge.

For every repeat, 8,000 development points are generated and split into 7,200 training and 800 validation examples. A separate 10,000-point test set is generated from an independent seed and is evaluated only after the best-validation checkpoint has been restored.

The MLP recipe was selected at three hidden layers of width 100. In the paper profile, that same recipe is transferred unchanged across one to four hidden layers and widths 5, 10, 20, 50, 100, 200 and 300. Network size is the comparison variable; it is not quietly retuned for each point.

In [ ]:
# -----------------------------------------------------------------------------
# USER SETTINGS
# -----------------------------------------------------------------------------

RUN_PROFILE = "quick"       # "quick" or "paper"
RUN_KAN = True
RUN_MLP_BASELINES = True

TWIN_PATH = Path(
    os.environ.get("SYNE_TWIN_PATH", Path.cwd() / "B1_MLP_3_50_50_1.pt")
).expanduser()

OUTPUT_ROOT = Path(
    os.environ.get(
        "SYNE_RESULTS_DIR",
        f"results_2d_classification_release_{RUN_PROFILE}",
    )
).expanduser()

PROFILES = {
    "quick": {
        "max_epochs": 120,
        "evaluation_seeds": [404],
        # Keep the tuned 3x100 MLP in the quick run so the final plotting cell
        # exercises the same path used by the paper confirmation.
        "mlp_shapes": [(1, 10), (2, 10), (3, 100)],
    },
    "paper": {
        "max_epochs": 800,
        "evaluation_seeds": [404, 505, 606],
        "mlp_shapes": [
            (depth, width)
            for depth in (1, 2, 3, 4)
            for width in (5, 10, 20, 50, 100, 200, 300)
        ],
    },
}

if RUN_PROFILE not in PROFILES:
    raise ValueError(f"RUN_PROFILE must be one of {list(PROFILES)}")

PROFILE = PROFILES[RUN_PROFILE]
MAX_EPOCHS = int(PROFILE["max_epochs"])
EVALUATION_SEEDS = list(PROFILE["evaluation_seeds"])
MLP_SHAPES = list(PROFILE["mlp_shapes"])

TASKS = ["checker_5x5", "yin_yang"]
TASK_LABELS = {
    "checker_5x5": "5×5 checkerboard",
    "yin_yang": "Five-circle yin–yang",
}
TASK_ARCHITECTURES = {
    "checker_5x5": [2, 6, 1],
    "yin_yang": [2, 2, 1],
}

DEVICES_PER_EDGE = 8
N_INPUTS = 2
DEVELOPMENT_POINTS = 8000
TEST_POINTS = 10000
VALIDATION_FRACTION = 0.10
VISUAL_MLP_SHAPE = (3, 100)

if VISUAL_MLP_SHAPE not in MLP_SHAPES:
    raise ValueError(
        f"VISUAL_MLP_SHAPE={VISUAL_MLP_SHAPE} must be included in MLP_SHAPES"
    )

print("Run profile:", RUN_PROFILE)
print("Twin checkpoint:", TWIN_PATH)
print("Results folder:", OUTPUT_ROOT)
print("Tasks:", TASKS)
print("KAN architectures:", TASK_ARCHITECTURES)
print("Evaluation seeds:", EVALUATION_SEEDS)
print("MLP shapes:", MLP_SHAPES)

## Exact classification tasks and data protocol

The checkerboard is formed directly on $[-1,1]^2$ by alternating labels over a $5\times5$ square grid.

The yin–yang target is the archived five-circle construction used for the paper: two radius-0.5 lobes, two radius-0.2 colour-reversal dots, and the remaining side regions split left/right. Sampling with $r=\sqrt{u}$ gives a uniform point density over the unit disk.

The preview below is not decoration. It is the quickest way of catching a broken label convention, a non-uniform disk sampler or an accidental change to the checkerboard indexing before several hours of training are wasted.

In [ ]:
def sample_unit_disk(rng, number_of_points):
    radius = np.sqrt(rng.random(number_of_points))
    angle = rng.uniform(0.0, 2.0 * np.pi, number_of_points)
    return np.column_stack(
        [radius * np.cos(angle), radius * np.sin(angle)]
    ).astype(np.float32)


def yin_yang_labels(points):
    x, y = points[:, 0], points[:, 1]

    in_top_circle = x**2 + (y - 0.5) ** 2 <= 0.5**2
    in_bottom_circle = x**2 + (y + 0.5) ** 2 <= 0.5**2
    in_top_dot = x**2 + (y - 0.5) ** 2 <= 0.2**2
    in_bottom_dot = x**2 + (y + 0.5) ** 2 <= 0.2**2

    labels = np.zeros(len(points), dtype=np.float32)
    labels[in_bottom_circle] = 1.0

    side_region = ~in_top_circle & ~in_bottom_circle
    labels[side_region & (x >= 0.0)] = 1.0

    # The two small dots reverse the colour of the surrounding lobe.
    labels[in_top_dot] = 1.0
    labels[in_bottom_dot] = 0.0
    return labels


def generate_task(task_name, seed, number_of_points):
    rng = np.random.default_rng(seed)

    if task_name == "checker_5x5":
        points = rng.uniform(
            -1.0, 1.0, (number_of_points, 2)
        ).astype(np.float32)
        square = np.floor((points + 1.0) * 2.5).astype(int).clip(0, 4)
        labels = ((square[:, 0] + square[:, 1]) % 2).astype(np.float32)
        return points, labels

    if task_name == "yin_yang":
        points = sample_unit_disk(rng, number_of_points)
        return points, yin_yang_labels(points)

    raise KeyError(f"Unknown classification task: {task_name}")


def stratified_train_validation_indices(
    x,
    y,
    seed,
    validation_fraction=VALIDATION_FRACTION,
):
    del x  # retained in the signature to make the split call self-documenting
    rng = np.random.default_rng(seed)
    training_indices, validation_indices = [], []

    for class_label in (0, 1):
        indices = np.flatnonzero(y == class_label)
        rng.shuffle(indices)
        number_validation = int(round(validation_fraction * len(indices)))
        validation_indices.extend(indices[:number_validation])
        training_indices.extend(indices[number_validation:])

    rng.shuffle(training_indices)
    rng.shuffle(validation_indices)
    return np.asarray(training_indices), np.asarray(validation_indices)


def make_split(task_name, seed):
    development_x, development_y = generate_task(
        task_name,
        stable_seed("development_pool", task_name, seed),
        DEVELOPMENT_POINTS,
    )
    test_x, test_y = generate_task(
        task_name,
        stable_seed("held_out_test", task_name, seed),
        TEST_POINTS,
    )

    training_indices, validation_indices = (
        stratified_train_validation_indices(
            development_x,
            development_y,
            seed,
        )
    )

    return {
        "x_train": torch.from_numpy(development_x[training_indices]),
        "y_train": torch.from_numpy(development_y[training_indices, None]),
        "x_val": torch.from_numpy(development_x[validation_indices]),
        "y_val": torch.from_numpy(development_y[validation_indices, None]),
        "x_test": torch.from_numpy(test_x),
        "y_test": torch.from_numpy(test_y[:, None]),
    }


fig, axes = plt.subplots(
    1, 2, figsize=(9.2, 4.1), constrained_layout=True
)
for axis, task_name in zip(axes, TASKS):
    points, labels = generate_task(
        task_name,
        stable_seed("preview", task_name),
        5000,
    )
    axis.scatter(
        points[:, 0],
        points[:, 1],
        c=labels,
        cmap="bwr",
        s=5,
        alpha=0.75,
    )
    axis.set_title(TASK_LABELS[task_name])
    axis.set_xlim(-1, 1)
    axis.set_ylim(-1, 1)
    axis.set_aspect("equal")
    axis.set_xlabel("x")
    axis.set_ylabel("y")

fig.suptitle("Released 2D classification targets")
plt.show()

## Frozen B1 digital twin and physical KAN

Each KAN edge is a parallel bank of SYNE responses predicted by the frozen B1 digital twin. The twin consumes one signal voltage and two constant control voltages and returns one predicted current. Its weights never change in this notebook.

The signal and control sigmoids are bounded physical parameterisations, not temporary training nonlinearities. They map unconstrained trainable variables onto the normalised device ranges and remain active during training, validation and inference.

There are no KAN range penalties, projection steps, clamps, weight-decay terms or extra control-voltage initialisation limits in the release recipe.

The implementation retains one output-bias variable per simulated SYNE contribution because that is the representation used in the completed runs. Biases arriving at the same neuron are algebraically consolidated for the manuscript count:

$$N_{\mathrm{paper}}=5D\sum_l n_l n_{l+1}+\sum_l n_{l+1}.$$

This gives 727 parameters for the checkerboard KAN and 243 for the yin–yang KAN.

In [ ]:
class RebuiltTwin(nn.Module):
    """Fallback for a checkpoint containing only an ordinary MLP state_dict."""
    def __init__(self, weights, biases):
        super().__init__()
        self.weights = nn.ParameterList([nn.Parameter(w.clone(), requires_grad=False) for w in weights])
        self.biases = nn.ParameterList([nn.Parameter(b.clone(), requires_grad=False) for b in biases])

    def forward(self, x):
        for i, (weight, bias) in enumerate(zip(self.weights, self.biases)):
            x = F.linear(x, weight, bias)
            if i + 1 < len(self.weights):
                x = F.relu(x)
        return x

def _natural_key(name: str):
    import re
    return [int(p) if p.isdigit() else p for p in re.split(r"(\d+)", name)]

def _strip_prefix(text: str, prefix: str) -> str:
    """Python 3.8-compatible replacement for str.removeprefix()."""
    return text[len(prefix):] if text.startswith(prefix) else text

def load_frozen_twin(path: Path) -> nn.Module:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Digital twin not found: {path}\n"
            "Set TWIN_PATH in the configuration cell. The expected checkpoint is either "
            "a saved nn.Module, [module, x_scale, y_scale], or an MLP state_dict."
        )
    try:
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        # Compatibility with PyTorch versions predating `weights_only`.
        payload = torch.load(path, map_location=DEVICE)
    if isinstance(payload, nn.Module):
        twin = payload
    elif isinstance(payload, (list, tuple)) and payload and isinstance(payload[0], nn.Module):
        twin = payload[0]
    else:
        state = payload.get("model_state_dict", payload.get("state_dict", payload)) if isinstance(payload, dict) else None
        if not isinstance(state, dict):
            raise TypeError(f"Unsupported twin checkpoint payload: {type(payload)}")
        cleaned_state = {}
        for key, value in state.items():
            key = _strip_prefix(str(key), "module.")
            key = _strip_prefix(key, "model.")
            cleaned_state[key] = value
        state = cleaned_state
        weight_keys = sorted([k for k, v in state.items() if k.endswith("weight") and getattr(v, "ndim", 0) == 2], key=_natural_key)
        if not weight_keys:
            raise ValueError("No linear weights found in the twin state_dict")
        weights, biases = [], []
        for key in weight_keys:
            weights.append(state[key].float())
            bias_key = key[:-6] + "bias"
            biases.append(state.get(bias_key, torch.zeros(state[key].shape[0])).float())
        twin = RebuiltTwin(weights, biases)

    twin = twin.to(DEVICE).float().eval()
    for parameter in twin.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad():
        probe = torch.zeros(8, 3, device=DEVICE)
        out = twin(probe).reshape(-1)
    if out.numel() != 8 or not bool(torch.isfinite(out).all()):
        raise ValueError("The twin failed a finite 8x3 input/output probe")
    print("Loaded and froze twin:", path, "|", type(twin).__name__)
    return twin


class PhysicalKAN(nn.Module):
    """Constant-control physical SYNE-KAN used across the release notebooks."""
    def __init__(self, twin, architecture, devices_per_edge, hp):
        super().__init__()
        self.twin = twin
        self.architecture = list(map(int, architecture))
        self.devices_per_edge = int(devices_per_edge)
        self.hp = dict(hp)
        self.controls_per_device = 2
        self.squash_gain = 6.0
        self.output_gain_scale = float(hp["output_gain_scale"])
        self.twin_microbatch = int(hp.get("twin_microbatch", 262144))
        for p in self.twin.parameters():
            p.requires_grad_(False)

        self.S_min, self.S_max = nn.ParameterList(), nn.ParameterList()
        self.V, self.G_o = nn.ParameterList(), nn.ParameterList()
        self.B_o = nn.ParameterList()
        for n_in, n_out in zip(self.architecture[:-1], self.architecture[1:]):
            shape = (n_in, n_out, self.devices_per_edge)
            self.S_min.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.S_max.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.V.append(nn.Parameter(torch.zeros(*shape, 2, device=DEVICE)))
            self.G_o.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.B_o.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
        self._initialise()

    def train(self, mode=True):
        super().train(mode)
        self.twin.eval()
        return self

    def squash_signal(self, raw):
        return 2.0 * torch.sigmoid(self.squash_gain * raw) - 1.0

    @staticmethod
    def squash_control(raw):
        return 2.0 * torch.sigmoid(raw) - 1.0

    def inverse_signal(self, value):
        p = ((value + 1.0) * 0.5).clamp(1e-6, 1.0 - 1e-6)
        return torch.log(p / (1.0 - p)) / self.squash_gain

    def _initialise(self):
        with torch.no_grad():
            for layer, (n_in, n_out) in enumerate(zip(self.architecture[:-1], self.architecture[1:])):
                shape = (n_in, n_out, self.devices_per_edge)
                count = int(np.prod(shape))
                dv_min = float(self.hp["init_dv_min"])
                dv_max = min(float(self.hp["init_dv_max"]), 1.96)
                widths = torch.linspace(dv_min, dv_max, count, device=DEVICE)
                step = (dv_max - dv_min) / max(count - 1, 1)
                widths += (torch.rand(count, device=DEVICE) - 0.5) * 2 * step * float(self.hp["span_jitter"])
                widths = widths.clamp(dv_min, dv_max)[torch.randperm(count, device=DEVICE)].reshape(shape)
                fan = float(n_in * self.devices_per_edge)
                if self.hp["init_style"] == "powerlaw":
                    widths = dv_min + (widths - dv_min) * max(0.1, min(1.0, fan ** (-float(self.hp["pl_beta"]))))
                inverted = torch.rand(shape, device=DEVICE) < float(self.hp["invert_probability"])
                low, high = -0.98 + 0.5 * widths, 0.98 - 0.5 * widths
                centre = low + torch.rand(shape, device=DEVICE) * (high - low)
                centre = torch.where(inverted, torch.zeros_like(centre), centre)
                vmin, vmax = centre - 0.5 * widths, centre + 0.5 * widths
                lo = torch.where(inverted, vmax, vmin)
                hi = torch.where(inverted, vmin, vmax)
                self.S_min[layer].copy_(self.inverse_signal(lo.clamp(-0.98, 0.98)))
                self.S_max[layer].copy_(self.inverse_signal(hi.clamp(-0.98, 0.98)))

                shrink = fan ** (-float(self.hp["pl_alpha"])) if self.hp["init_style"] == "powerlaw" else 1.0
                limit = math.sqrt(6.0 / 4.0) * float(self.hp["control_init_scale"]) * shrink
                # No lower/upper control-voltage initialisation limit. The raw
                # parameters are sampled symmetrically, then mapped to the
                # normalised physical voltage by squash_control() in forward().
                self.V[layer].uniform_(-limit, limit)
                self.G_o[layer].zero_()
                self.B_o[layer].zero_()

    def _twin_forward(self, flat):
        return torch.cat([self.twin(flat[i:i+self.twin_microbatch]).reshape(-1)
                          for i in range(0, len(flat), self.twin_microbatch)], dim=0)

    def forward(self, inputs):
        activation = inputs
        for layer, (n_in, n_out) in enumerate(zip(self.architecture[:-1], self.architecture[1:])):
            scalar = activation.unsqueeze(2).unsqueeze(3)
            pmin = self.squash_signal(self.S_min[layer])
            pmax = self.squash_signal(self.S_max[layer])
            signal = (0.5 * (pmax-pmin).unsqueeze(0) * scalar + 0.5 * (pmax+pmin).unsqueeze(0)).unsqueeze(-1)
            controls = self.squash_control(self.V[layer]).unsqueeze(0).expand(len(activation), -1, -1, -1, -1)
            twin_in = torch.cat([signal, controls], dim=-1)
            twin_out = self._twin_forward(twin_in.reshape(-1, 3)).reshape(
                len(activation), n_in, n_out, self.devices_per_edge)
            contribution = self.output_gain_scale * self.G_o[layer].unsqueeze(0) * twin_out
            contribution = contribution + self.B_o[layer].unsqueeze(0)
            activation = contribution.sum(dim=(1, 3))
        return activation

    def parameter_groups(self):
        ranges = list(self.S_min.parameters()) + list(self.S_max.parameters())
        controls = list(self.V.parameters())
        outputs = list(self.G_o.parameters()) + list(self.B_o.parameters())
        return [
            {"name": "gains_biases", "params": outputs, "lr": float(self.hp["lr_gobo"]), "weight_decay": 0.0},
            {"name": "controls", "params": controls, "lr": float(self.hp["lr_v"]), "weight_decay": 0.0},
            {"name": "ranges", "params": ranges, "lr": float(self.hp["lr_s"]), "weight_decay": 0.0},
        ]

    def actual_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def manuscript_parameter_count(self):
        edges = sum(a*b for a, b in zip(self.architecture[:-1], self.architecture[1:]))
        post_synaptic = sum(self.architecture[1:])
        return 5 * edges * self.devices_per_edge + post_synaptic

## ReLU MLP baseline and shared training loop

The baseline is an ordinary fully connected ReLU network with Xavier-initialised weights and zero biases. Both model families use binary cross-entropy with logits, the same generated data, the same train/validation/test protocol and the same maximum epoch budget.

Validation is checked every ten epochs. The checkpoint with the lowest validation BCE is restored before validation accuracy and held-out test accuracy are calculated.

The KAN uses Adam with three learning-rate groups: output gains/biases, control voltages and signal ranges. The MLP uses the fixed optimiser recipe selected in the development notebook.

In [ ]:
class ReLUMLP(nn.Module):
    def __init__(self, n_in, n_out, width, depth):
        super().__init__()
        dims = [n_in] + [int(width)] * int(depth) + [n_out]
        layers = []
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            linear = nn.Linear(a, b)
            nn.init.xavier_uniform_(linear.weight)
            nn.init.zeros_(linear.bias)
            layers.append(linear)
            if i + 1 < len(dims) - 1:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def actual_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

def compact_state(model):
    trainable = {name for name, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in trainable}

def restore_compact_state(model, state):
    current = model.state_dict()
    current.update({k: v.to(current[k].device) for k, v in state.items()})
    model.load_state_dict(current, strict=True)

def make_optimizer(model, hp, is_kan):
    if is_kan:
        return torch.optim.Adam(
            model.parameter_groups(),
            betas=(float(hp["beta1"]), float(hp["beta2"])),
            eps=float(hp.get("adam_eps", 1e-8)),
        )
    cls = torch.optim.AdamW if hp["optimizer"] == "adamw" else torch.optim.Adam
    return cls(model.parameters(), lr=float(hp["lr"]),
               betas=(float(hp["beta1"]), float(hp["beta2"])),
               eps=1e-8, weight_decay=float(hp["weight_decay"]))

def make_scheduler(optimizer, hp, max_epochs):
    if hp["scheduler"] == "cosine":
        minimum = float(hp["lr_min_frac"])
        def cosine_multiplier(epoch):
            progress = min(max(epoch, 0), max_epochs - 1) / float(max(max_epochs - 1, 1))
            return minimum + 0.5 * (1.0 - minimum) * (1.0 + math.cos(math.pi * progress))
        # LambdaLR multiplies every group's own base LR by the same factor,
        # preserving the NASA lr_gobo : lr_v : lr_s ratios throughout training.
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=cosine_multiplier)
    if hp["scheduler"] == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(20, max_epochs // 8), gamma=float(hp["step_gamma"]))
    return None

@torch.no_grad()
def predict(model, x, chunk=8192):
    model.eval()
    outs = []
    for i in range(0, len(x), chunk):
        outs.append(model(x[i:i+chunk].to(DEVICE)).detach().cpu())
    return torch.cat(outs)

def train_one(model, split, hp, problem, max_epochs, seed, verbose=False):
    seed_everything(seed)
    model = model.to(DEVICE)
    optimizer = make_optimizer(model, hp, isinstance(model, PhysicalKAN))
    scheduler = make_scheduler(optimizer, hp, max_epochs)
    loss_fn = nn.BCEWithLogitsLoss() if problem == "classification" else nn.MSELoss()
    loader = DataLoader(TensorDataset(split["x_train"], split["y_train"]),
                        batch_size=int(hp["batch_size"]), shuffle=True,
                        generator=torch.Generator().manual_seed(seed), pin_memory=torch.cuda.is_available())
    best_loss, best_epoch, best_state = float("inf"), 0, None
    evaluations_without_improvement = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred.reshape_as(yb), yb)
            loss.backward()
            if float(hp["grad_clip"]) > 0:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(hp["grad_clip"]))
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        if epoch == 1 or epoch % int(hp["evaluate_every"]) == 0 or epoch == max_epochs:
            val_pred = predict(model, split["x_val"])
            val_loss = float(loss_fn(val_pred.reshape_as(split["y_val"]), split["y_val"]).item())
            history.append((epoch, val_loss))
            if val_loss < best_loss - float(hp["min_delta"]):
                best_loss, best_epoch, best_state = val_loss, epoch, compact_state(model)
                evaluations_without_improvement = 0
            else:
                evaluations_without_improvement += 1
            if verbose:
                print(f"epoch={epoch:4d} val_loss={val_loss:.6g} best={best_loss:.6g}@{best_epoch}")
            if epoch >= int(hp["minimum_epochs"]) and evaluations_without_improvement >= int(hp["patience_evaluations"]):
                break
    restore_compact_state(model, best_state)
    val_pred = predict(model, split["x_val"])
    if problem == "classification":
        val_metric = float(((val_pred.reshape(-1) >= 0) == (split["y_val"].reshape(-1) >= 0.5)).float().mean())
    else:
        val_metric = float(F.mse_loss(val_pred.reshape_as(split["y_val"]), split["y_val"]))
    return model, {"val_loss": best_loss, "val_metric": val_metric, "best_epoch": best_epoch, "history": history}

## Fixed release recipes

These are the completed Optuna recipes, decoded into ordinary dictionaries so that the release does not depend on an Optuna installation or a private SQLite study.

The shared KAN recipe is trial 107 from the corrected task-specific architecture study. Its tuning objective was a mean validation classification error of `7.916679 × 10⁻³` across both tasks and three matched repeats.

The MLP recipe is trial 22 from the fixed `[2,100,100,100,1]` optimiser study, with a mean validation error of `6.458352 × 10⁻³`.

As a scale check, fresh 800-epoch confirmation gave:

- physical KAN: `98.43 ± 0.22%` test accuracy on the checkerboard and `99.08 ± 0.19%` on yin–yang;
- three-layer width-100 MLP: `98.61 ± 0.13%` and `99.37 ± 0.06%`.

Those values are reference results, not assertions that every PyTorch/CUDA combination must agree in the last decimal place.

In [ ]:
KAN_RECIPE = {
    "batch_size": 64,
    "beta1": 0.8732514841987591,
    "beta2": 0.9319380888083301,
    "control_init_scale": 0.37017654055169313,
    "evaluate_every": 10,
    "grad_clip": 1.1216607973942743,
    "init_dv_min": 0.9217543569919135,
    "init_dv_max": 1.4779169762511962,
    "init_style": "powerlaw",
    "invert_probability": 0.17769176737480483,
    "lr_gobo": 0.004351773638989695,
    "lr_min_frac": 0.017159459089982974,
    "lr_s": 0.00043962166763540476,
    "lr_v": 9.058647273851198e-05,
    "min_delta": 1e-6,
    "minimum_epochs": 100,
    "output_gain_scale": 50.0,
    "patience_evaluations": 20,
    "pl_alpha": 1.2699093429385424,
    "pl_beta": 1.4623654755469275,
    "scheduler": "cosine",
    "span_jitter": 0.053076231040254655,
    "step_gamma": 0.8239896378821361,
    "adam_eps": 1e-8,
}

MLP_RECIPE = {
    "batch_size": 128,
    "beta1": 0.8998409800017048,
    "beta2": 0.9533384450257022,
    "evaluate_every": 10,
    "grad_clip": 0.46328999985200336,
    "lr": 0.003343113573826987,
    "lr_min_frac": 0.021685662838299456,
    "min_delta": 1e-6,
    "minimum_epochs": 100,
    "optimizer": "adam",
    "patience_evaluations": 20,
    "scheduler": "step",
    "step_gamma": 0.5976125383564541,
    "weight_decay": 2.9895833789873958e-09,
}

RECIPE_PROVENANCE = {
    "kan": {
        "study": "syne_classification_v3_task_arch_d8_nopenalty_groups_kan_full",
        "best_trial": 107,
        "mean_validation_error": 0.00791667898495992,
    },
    "mlp": {
        "study": "syne_classification_v3_task_arch_d8_nopenalty_groups_mlp_full",
        "best_trial": 22,
        "mean_validation_error": 0.006458352009455363,
        "tuning_architecture": [2, 100, 100, 100, 1],
    },
}

CONFIRMATION_REFERENCE = {
    "kan": {
        "checker_5x5_test_accuracy_mean": 0.984333336353302,
        "checker_5x5_test_accuracy_std": 0.0021733249302264886,
        "yin_yang_test_accuracy_mean": 0.9907999833424886,
        "yin_yang_test_accuracy_std": 0.001931335738775039,
    },
    "mlp_3x100": {
        "checker_5x5_test_accuracy_mean": 0.9860999981562296,
        "checker_5x5_test_accuracy_std": 0.0012529950233879422,
        "yin_yang_test_accuracy_mean": 0.9937000075976053,
        "yin_yang_test_accuracy_std": 0.0006000101568781818,
    },
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)

TWIN = load_frozen_twin(TWIN_PATH)

print("\nPhysical KAN preflight:")
for task in TASKS:
    architecture = TASK_ARCHITECTURES[task]
    seed_everything(stable_seed("preflight", task))
    probe = PhysicalKAN(
        TWIN,
        architecture,
        DEVICES_PER_EDGE,
        KAN_RECIPE,
    )
    forward_probe = probe(torch.zeros(4, N_INPUTS, device=DEVICE))
    if tuple(forward_probe.shape) != (4, 1):
        raise AssertionError(
            f"Unexpected output shape for {task}: {tuple(forward_probe.shape)}"
        )
    if not bool(torch.isfinite(forward_probe).all()):
        raise FloatingPointError(f"Non-finite preflight output for {task}")

    print(
        f"  {task:12s} {architecture}, D={DEVICES_PER_EDGE}: "
        f"paper={probe.manuscript_parameter_count()}, "
        f"implementation={probe.actual_trainable_parameters()}"
    )
    del probe, forward_probe

release_manifest = {
    "run_profile": RUN_PROFILE,
    "tasks": TASKS,
    "task_labels": TASK_LABELS,
    "task_architectures": TASK_ARCHITECTURES,
    "devices_per_edge": DEVICES_PER_EDGE,
    "development_points": DEVELOPMENT_POINTS,
    "validation_fraction": VALIDATION_FRACTION,
    "test_points": TEST_POINTS,
    "evaluation_seeds": EVALUATION_SEEDS,
    "max_epochs": MAX_EPOCHS,
    "mlp_shapes": MLP_SHAPES,
    "kan_recipe": KAN_RECIPE,
    "mlp_recipe": MLP_RECIPE,
    "recipe_provenance": RECIPE_PROVENANCE,
    "confirmation_reference": CONFIRMATION_REFERENCE,
    "torch_version": torch.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "device": str(DEVICE),
}
(OUTPUT_ROOT / "release_manifest.json").write_text(
    json.dumps(release_manifest, indent=2),
    encoding="utf-8",
)

(OUTPUT_ROOT / "fixed_release_recipes.json").write_text(
    json.dumps(
        {
            "kan_recipe": KAN_RECIPE,
            "mlp_recipe": MLP_RECIPE,
            "provenance": RECIPE_PROVENANCE,
        },
        indent=2,
    ),
    encoding="utf-8",
)

## Resumable result logging

Each run ID contains the model family, task, architecture/size, recipe signature and epoch budget. A short quick-profile result therefore cannot be mistaken for a completed paper run.

The CSV is replaced atomically after every model. Predictions are saved for the two paper KANs and the three-layer width-100 MLP on the first evaluation seed, so the final plots never need to retrain a model.

In [ ]:
RESULTS_PATH = OUTPUT_ROOT / "runs.csv"


def recipe_signature(hp: dict) -> str:
    payload = json.dumps(hp, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


def prediction_path(
    model_family,
    task,
    configuration,
    seed,
    signature,
):
    filename = (
        f"{model_family}_{task}_{configuration}_seed{seed}_{signature}.npz"
    )
    return PREDICTION_ROOT / filename.replace("/", "_")


def load_results() -> pd.DataFrame:
    if not RESULTS_PATH.exists():
        return pd.DataFrame()
    frame = pd.read_csv(RESULTS_PATH)
    if "run_id" not in frame:
        raise ValueError(
            f"Existing results file has no run_id column: {RESULTS_PATH}"
        )
    return (
        frame.drop_duplicates("run_id", keep="last")
        .reset_index(drop=True)
    )


def save_results(frame: pd.DataFrame) -> None:
    temporary = RESULTS_PATH.with_suffix(".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(RESULTS_PATH)


RESULTS = load_results()
COMPLETED_RUN_IDS = (
    set(RESULTS["run_id"].astype(str)) if len(RESULTS) else set()
)


def run_and_record(
    *,
    model_family: str,
    task: str,
    seed: int,
    configuration: str,
    architecture: str,
    hp: dict,
    model_factory,
    devices_per_edge=None,
    depth=None,
    width=None,
    paper_parameters=None,
    implementation_parameters=None,
    save_prediction=False,
) -> bool:
    global RESULTS

    signature = recipe_signature(hp)
    run_id = "|".join(
        map(
            str,
            [
                model_family,
                task,
                seed,
                configuration,
                signature,
                MAX_EPOCHS,
            ],
        )
    )
    if run_id in COMPLETED_RUN_IDS:
        return False

    # This is the exact seed construction used for the fresh confirmation in
    # the development notebook. It also keeps the randomness matched across
    # MLP sizes instead of handing each architecture a different lucky draw.
    model_kind = model_family.lower()
    run_seed = stable_seed(
        "fresh_confirmation",
        model_kind,
        task,
        seed,
    )
    seed_everything(run_seed)  # seed before constructing the model
    model = model_factory()
    split = make_split(task, seed)

    started = time.time()
    model, training = train_one(
        model,
        split,
        hp,
        "classification",
        MAX_EPOCHS,
        run_seed,
    )
    elapsed_seconds = time.time() - started

    test_logits = predict(model, split["x_test"])
    test_bce = float(
        F.binary_cross_entropy_with_logits(
            test_logits.reshape_as(split["y_test"]),
            split["y_test"],
        ).item()
    )
    test_prediction = (test_logits.reshape(-1) >= 0.0).to(torch.int64)
    test_target = (split["y_test"].reshape(-1) >= 0.5).to(torch.int64)
    test_accuracy = float(
        (test_prediction == test_target).float().mean().item()
    )

    if paper_parameters is None:
        paper_parameters = model.actual_trainable_parameters()
    if implementation_parameters is None:
        implementation_parameters = model.actual_trainable_parameters()

    row = {
        "run_id": run_id,
        "recipe_signature": signature,
        "profile": RUN_PROFILE,
        "model_family": model_family,
        "task": task,
        "seed": int(seed),
        "training_seed": int(run_seed),
        "configuration": configuration,
        "architecture": architecture,
        "devices_per_edge": devices_per_edge,
        "depth": depth,
        "width": width,
        "trainable_parameters": int(paper_parameters),
        "implementation_parameters": int(implementation_parameters),
        "best_validation_bce": float(training["val_loss"]),
        "validation_accuracy": float(training["val_metric"]),
        "test_bce": test_bce,
        "test_accuracy": test_accuracy,
        "test_error": 1.0 - test_accuracy,
        "best_epoch": int(training["best_epoch"]),
        "max_epochs": MAX_EPOCHS,
        "elapsed_seconds": elapsed_seconds,
    }

    if save_prediction:
        np.savez_compressed(
            prediction_path(
                model_family,
                task,
                configuration,
                seed,
                signature,
            ),
            x=split["x_test"].numpy(),
            target=test_target.numpy(),
            logit=test_logits.numpy().reshape(-1),
            prediction=test_prediction.numpy(),
        )

    RESULTS = pd.concat(
        [RESULTS, pd.DataFrame([row])],
        ignore_index=True,
    )
    COMPLETED_RUN_IDS.add(run_id)
    save_results(RESULTS)

    print(
        f"{model_family:3s} | {configuration:16s} | "
        f"{task:12s} | seed {seed} | "
        f"test accuracy {100.0 * test_accuracy:6.3f}%"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return True

## Run the two physical KANs

The paper KANs are run first. They share one frozen recipe and eight SYNEs per edge, but retain their task-specific architectures.

Nothing is selected using the test accuracy. Each model is trained on the development set, checkpointed using validation BCE, restored to its strongest validation state and then evaluated once on the independent test set.

In [ ]:
if RUN_KAN:
    for task in TASKS:
        architecture = TASK_ARCHITECTURES[task]
        architecture_tag = "_".join(map(str, architecture))
        configuration = (
            f"KAN_{architecture_tag}_D{DEVICES_PER_EDGE}"
        )

        seed_everything(stable_seed("parameter_count", task))
        probe = PhysicalKAN(
            TWIN,
            architecture,
            DEVICES_PER_EDGE,
            KAN_RECIPE,
        )
        paper_parameters = probe.manuscript_parameter_count()
        implementation_parameters = probe.actual_trainable_parameters()
        del probe

        for seed in EVALUATION_SEEDS:
            run_and_record(
                model_family="KAN",
                task=task,
                seed=seed,
                configuration=configuration,
                architecture=str(architecture).replace(" ", ""),
                hp=KAN_RECIPE,
                model_factory=lambda a=architecture: PhysicalKAN(
                    TWIN,
                    a,
                    DEVICES_PER_EDGE,
                    KAN_RECIPE,
                ),
                devices_per_edge=DEVICES_PER_EDGE,
                paper_parameters=paper_parameters,
                implementation_parameters=implementation_parameters,
                save_prediction=(seed == EVALUATION_SEEDS[0]),
            )

## Run the ReLU MLP size sweep

Every MLP uses the same optimiser recipe. Only depth and width change. This is the point of the comparison: parameter count and architecture are exposed on the x-axis rather than absorbed into another tuning budget.

The three-hidden-layer width-100 model is retained for the final decision plots because it is the exact architecture used to select and confirm the released MLP recipe.

In [ ]:
if RUN_MLP_BASELINES:
    for depth, width in MLP_SHAPES:
        probe = ReLUMLP(N_INPUTS, 1, width, depth)
        parameter_count = probe.actual_trainable_parameters()
        del probe

        configuration = f"MLP_L{depth}_H{width}"
        architecture = (
            "["
            + ",".join(
                map(str, [N_INPUTS] + [width] * depth + [1])
            )
            + "]"
        )

        for task in TASKS:
            for seed in EVALUATION_SEEDS:
                run_and_record(
                    model_family="MLP",
                    task=task,
                    seed=seed,
                    configuration=configuration,
                    architecture=architecture,
                    hp=MLP_RECIPE,
                    model_factory=lambda d=depth, w=width: ReLUMLP(
                        N_INPUTS,
                        1,
                        w,
                        d,
                    ),
                    depth=depth,
                    width=width,
                    paper_parameters=parameter_count,
                    implementation_parameters=parameter_count,
                    save_prediction=(
                        (depth, width) == VISUAL_MLP_SHAPE
                        and seed == EVALUATION_SEEDS[0]
                    ),
                )


expected_runs = (
    (len(TASKS) if RUN_KAN else 0)
    + (
        len(MLP_SHAPES) * len(TASKS)
        if RUN_MLP_BASELINES
        else 0
    )
) * len(EVALUATION_SEEDS)

print(
    f"Completed rows in this result folder: "
    f"{len(load_results())} / {expected_runs}"
)

## Aggregate and plot the results

A configuration is included only when every requested evaluation seed has completed. Partly finished runs remain in `runs.csv`, but they are excluded from the summary plot rather than being allowed to acquire a misleadingly narrow error bar.

The scaling figure uses classification error, $1-\mathrm{accuracy}$, because the networks operate close to unity accuracy and the remaining mistakes are the informative quantity. The two tasks are shown separately because their paper KANs have different architectures and parameter counts.

In [ ]:
run_table = load_results()
if run_table.empty:
    raise RuntimeError(
        "No completed runs were found. Run at least one model cell first."
    )

expected_seeds = len(EVALUATION_SEEDS)

coverage = (
    run_table.groupby(
        ["model_family", "task", "configuration"],
        dropna=False,
    )
    .agg(
        rows=("run_id", "size"),
        seeds=("seed", "nunique"),
    )
    .reset_index()
)

complete_configurations = coverage[
    (coverage["rows"] == expected_seeds)
    & (coverage["seeds"] == expected_seeds)
][["model_family", "task", "configuration"]]

complete_rows = run_table.merge(
    complete_configurations,
    on=["model_family", "task", "configuration"],
    how="inner",
)

summary_by_task = (
    complete_rows.groupby(
        ["model_family", "task", "configuration"],
        dropna=False,
    )
    .agg(
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_std=("test_accuracy", "std"),
        test_accuracy_min=("test_accuracy", "min"),
        test_accuracy_max=("test_accuracy", "max"),
        test_error_mean=("test_error", "mean"),
        test_error_min=("test_error", "min"),
        test_error_max=("test_error", "max"),
        validation_accuracy_mean=("validation_accuracy", "mean"),
        best_epoch_mean=("best_epoch", "mean"),
        architecture=("architecture", "first"),
        devices_per_edge=("devices_per_edge", "first"),
        depth=("depth", "first"),
        width=("width", "first"),
        trainable_parameters=("trainable_parameters", "first"),
        implementation_parameters=(
            "implementation_parameters",
            "first",
        ),
    )
    .reset_index()
)

coverage.to_csv(
    OUTPUT_ROOT / "rollout_completion_audit.csv",
    index=False,
)
summary_by_task.to_csv(
    OUTPUT_ROOT / "summary_by_task_configuration.csv",
    index=False,
)

print(
    summary_by_task.sort_values(
        ["task", "model_family", "depth", "trainable_parameters"],
        na_position="first",
    ).to_string(index=False)
)


colors = {
    "KAN": "#0b3c6f",
    1: "#f39c12",
    2: "#e67e22",
    3: "#c65d00",
    4: "#8f3f00",
}
markers = {
    "KAN": "X",
    1: "^",
    2: "s",
    3: "D",
    4: "o",
}

error_floor = 0.5 / float(TEST_POINTS)

fig, axes = plt.subplots(
    1,
    len(TASKS),
    figsize=(12.0, 4.8),
    constrained_layout=True,
)
if len(TASKS) == 1:
    axes = [axes]

for axis, task in zip(axes, TASKS):
    task_summary = summary_by_task[
        summary_by_task["task"] == task
    ]

    kan_part = task_summary[
        task_summary["model_family"] == "KAN"
    ].sort_values("trainable_parameters")
    if len(kan_part):
        y = kan_part["test_error_mean"].to_numpy(float)
        y_min = kan_part["test_error_min"].to_numpy(float)
        y_max = kan_part["test_error_max"].to_numpy(float)
        y_plot = np.maximum(y, error_floor)
        y_min_plot = np.maximum(y_min, error_floor)
        y_max_plot = np.maximum(y_max, error_floor)
        axis.errorbar(
            kan_part["trainable_parameters"],
            y_plot,
            yerr=np.vstack(
                [
                    y_plot - y_min_plot,
                    y_max_plot - y_plot,
                ]
            ),
            marker=markers["KAN"],
            color=colors["KAN"],
            linestyle="none",
            markersize=10,
            capsize=3,
            label="Physical SYNE-KAN",
        )

    for depth in (1, 2, 3, 4):
        mlp_part = task_summary[
            (task_summary["model_family"] == "MLP")
            & (task_summary["depth"] == depth)
        ].sort_values("trainable_parameters")
        if len(mlp_part):
            y = mlp_part["test_error_mean"].to_numpy(float)
            y_min = mlp_part["test_error_min"].to_numpy(float)
            y_max = mlp_part["test_error_max"].to_numpy(float)
            y_plot = np.maximum(y, error_floor)
            y_min_plot = np.maximum(y_min, error_floor)
            y_max_plot = np.maximum(y_max, error_floor)
            axis.errorbar(
                mlp_part["trainable_parameters"],
                y_plot,
                yerr=np.vstack(
                    [
                        y_plot - y_min_plot,
                        y_max_plot - y_plot,
                    ]
                ),
                marker=markers[depth],
                color=colors[depth],
                linestyle="--",
                linewidth=1.6,
                capsize=3,
                label=(
                    f"MLP, {depth} hidden "
                    f"layer{'s' if depth > 1 else ''}"
                ),
            )

    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel(
        "Network size / trainable parameters "
        "(paper count for KAN)"
    )
    axis.set_ylabel("Held-out test classification error")
    axis.set_title(TASK_LABELS[task])
    axis.grid(True, which="both", alpha=0.25)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    frameon=False,
)
fig.suptitle("2D classification: parameter–performance scaling")
fig.savefig(
    OUTPUT_ROOT / "parameter_error_scaling.png",
    dpi=240,
)
plt.show()

## Held-out decision plots

The final plots use saved predictions from the first evaluation seed. They show the ground-truth test labels beside the task-specific physical KAN and the tuned three-layer width-100 MLP.

These are direct checks on the geometry. A plausible scalar accuracy can still conceal a flipped label convention, a malformed disk boundary or a model that has failed in one compact part of the checkerboard.

In [ ]:
visual_seed = EVALUATION_SEEDS[0]
mlp_depth, mlp_width = VISUAL_MLP_SHAPE
mlp_configuration = f"MLP_L{mlp_depth}_H{mlp_width}"
mlp_signature = recipe_signature(MLP_RECIPE)
kan_signature = recipe_signature(KAN_RECIPE)

for task in TASKS:
    architecture = TASK_ARCHITECTURES[task]
    architecture_tag = "_".join(map(str, architecture))
    kan_configuration = (
        f"KAN_{architecture_tag}_D{DEVICES_PER_EDGE}"
    )

    kan_path = prediction_path(
        "KAN",
        task,
        kan_configuration,
        visual_seed,
        kan_signature,
    )
    mlp_path = prediction_path(
        "MLP",
        task,
        mlp_configuration,
        visual_seed,
        mlp_signature,
    )

    if not (kan_path.exists() and mlp_path.exists()):
        print("Skipping incomplete decision plot:", task)
        continue

    kan_data = np.load(kan_path)
    mlp_data = np.load(mlp_path)

    points = kan_data["x"]
    target = kan_data["target"].reshape(-1)
    kan_prediction = kan_data["prediction"].reshape(-1)
    mlp_prediction = mlp_data["prediction"].reshape(-1)

    kan_accuracy = float(np.mean(kan_prediction == target))
    mlp_accuracy = float(np.mean(mlp_prediction == target))

    labels_to_plot = [target, kan_prediction, mlp_prediction]
    titles = [
        "Ground truth",
        (
            f"Physical KAN {str(architecture).replace(' ', '')}, "
            f"D{DEVICES_PER_EDGE}\n"
            f"accuracy = {100.0 * kan_accuracy:.2f}%"
        ),
        (
            f"MLP L{mlp_depth}, H{mlp_width}\n"
            f"accuracy = {100.0 * mlp_accuracy:.2f}%"
        ),
    ]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12.0, 3.8),
        constrained_layout=True,
    )
    for axis, labels, title in zip(
        axes,
        labels_to_plot,
        titles,
    ):
        axis.scatter(
            points[:, 0],
            points[:, 1],
            c=labels,
            cmap="bwr",
            s=4,
            alpha=0.72,
        )
        axis.set_title(title)
        axis.set_aspect("equal")
        axis.set_xlim(-1, 1)
        axis.set_ylim(-1, 1)
        axis.set_xlabel("x")
        axis.set_ylabel("y")

    fig.suptitle(TASK_LABELS[task])
    fig.savefig(
        OUTPUT_ROOT / f"{task}_decision_boundaries.png",
        dpi=220,
    )
    plt.show()

## Output files

Each run profile writes to its own result directory:

- `runs.csv`: one row per trained model, task and repeat;
- `summary_by_task_configuration.csv`: repeat-aggregated accuracy and error for every complete configuration;
- `rollout_completion_audit.csv`: explicit coverage check for interrupted sweeps;
- `release_manifest.json`: tasks, architectures, seeds, software versions and fixed recipes;
- `fixed_release_recipes.json`: the final KAN and MLP hyperparameters with Optuna provenance;
- `predictions/*.npz`: held-out labels, logits and predictions for the paper KANs and the three-layer width-100 MLP;
- `parameter_error_scaling.png`: classification error against trainable parameter count;
- one held-out decision-boundary PNG per task.

The raw run table is the primary numerical record. Every summary and figure is regenerated from it; no result is buried only in notebook output.